In [1]:
!pip install pandas

**Cargar datos desde GitHub**

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/kevinnumanzor17/etl-seguros-pipeline/refs/heads/main/data/raw/tipos_seguro.csv"

df = pd.read_csv(url)

df.head()

,id_tipo_seguro,tipo,categoria,riesgo_base
0,1,Pyme,Familiar,-
1,2,Industrial,Empresarial,4.68
2,3,Industrial,Familiar,5.10
3,4,Industrial,Personal,NaN
4,5,Auto,empresarial,9.07


**Exploración**

In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_tipo_seguro  12 non-null     int64 
 1   tipo            12 non-null     object
 2   categoria       10 non-null     object
 3   riesgo_base     10 non-null     object
dtypes: int64(1), object(3)
memory usage: 516.0+ bytes


,0
id_tipo_seguro,0
tipo,0
categoria,2
riesgo_base,2


**Limpieza de datos**

In [4]:
tipos_seguro = df.copy()

# limpiar nombres de columnas
tipos_seguro.columns = tipos_seguro.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in tipos_seguro.select_dtypes(include='object').columns:
    tipos_seguro[col] = tipos_seguro[col].astype(str).str.strip()

# convertir vacíos en null
tipos_seguro = tipos_seguro.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
tipos_seguro = tipos_seguro.drop_duplicates()

**riesgo_base debe ser numérico**

In [5]:
tipos_seguro['riesgo_base'] = pd.to_numeric(
    tipos_seguro['riesgo_base'],
    errors='coerce'
)

**Separar válidos y rechazados**

In [6]:
validos = tipos_seguro[
    tipos_seguro['tipo'].notna() &
    tipos_seguro['categoria'].notna() &
    tipos_seguro['riesgo_base'].notna()
].copy()

rechazados = tipos_seguro[
    tipos_seguro['tipo'].isna() |
    tipos_seguro['categoria'].isna() |
    tipos_seguro['riesgo_base'].isna()
].copy()

**Motivo de rechazo**

In [7]:
def motivo(row):
    motivos = []

    if pd.isna(row['tipo']):
        motivos.append("tipo_vacio")

    if pd.isna(row['categoria']):
        motivos.append("categoria_vacio")

    if pd.isna(row['riesgo_base']):
        motivos.append("riesgo_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

**Exportar archivos**

In [8]:
validos.to_csv("tipos_seguro_curated.csv", index=False)
rechazados.to_csv("tipos_seguro_rejects.csv", index=False)